# Installing packages and loading data file

In [2]:
# Install packages 
#%pip install pandas numpy matplotlib scikit-learn ydata-profiling

In [1]:
# Import relavent packages 
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
import matplotlib.pyplot as plt
from scipy.stats import randint

In [2]:
# Load cleaned dataset
df_clean = pd.read_csv("/Users/muyuanli/Library/CloudStorage/OneDrive-CornellUniversity/Desktop/ANSC 6040/df_clean.csv")


# Splitting dataset for Random Forest
## Stratify by cow ID and split by time 

In [5]:
# Checking how many records on average a cow has
df_clean.groupby("AnimalId")["EventDate"].agg(["min", "max", "count"]).describe()

,count
count,9011.000000
mean,669.522916
std,448.709327
min,1.000000
25%,276.000000
50%,631.000000
75%,1045.500000
max,1673.000000


In [6]:
# Checking data type
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6033071 entries, 0 to 6033070
Data columns (total 14 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   AnimalId                float64
 1   LactationNumber         float64
 2   DaysInMilk              float64
 3   ReproductionStatus      object 
 4   EventDate               object 
 5   AvgMilkFlow             float64
 6   Flow30_60Session        float64
 7   YieldFirst2Min_Session  float64
 8   YieldSession            float64
 9   DurationSession_Sec     int64  
 10  Milking                 int64  
 11  z_AvgMilkFlow           float64
 12  z_Flow30_60Session      float64
 13  z_YieldSession          float64
dtypes: float64(10), int64(2), object(2)
memory usage: 644.4+ MB


In [3]:
# Convert variables to correct forms
df_clean["EventDate"] = pd.to_datetime(df_clean["EventDate"])
df_clean["AnimalId"] = df_clean["AnimalId"].astype(int)

# Subset for a smaller dataset

In [4]:
all_cows = df_clean["AnimalId"].unique()
rng = np.random.default_rng(seed=42)
sampled_cows = rng.choice(all_cows, size=int(len(all_cows) * 0.05), replace=False) 

df_small = df_clean[df_clean["AnimalId"].isin(sampled_cows)]
print(df_small.shape)
print(df_small["AnimalId"].nunique())

(311810, 14)
450


In [6]:
# Checking for cows with few records
counts = df_small["AnimalId"].value_counts()
print(counts.describe())
print((counts < 5).sum(), "cows with fewer than 5 records")

# Removing 2 cows with less than 5 records
valid_cows = counts[counts >= 5].index
df_small = df_small[df_small["AnimalId"].isin(valid_cows)]
print(df_small["AnimalId"].nunique(), "cows remaining")
print(df_small.shape)

count     448.000000
mean      695.995536
std       433.306382
min         5.000000
25%       316.000000
50%       692.000000
75%      1042.000000
max      1636.000000
Name: count, dtype: float64
0 cows with fewer than 5 records
448 cows remaining
(311806, 14)


In [7]:
# Sort dataset by cow ID then EventDate
df_sort = df_small.sort_values(["AnimalId", "EventDate"])

In [8]:
# Defining train/stest status for each cow record
def split_by_time(group, test_frac=0.2):
    n_test = max(1, int(len(group) * test_frac))
    return pd.Series(["train"] * (len(group) - n_test) + ["test"] * n_test,
                      index=group.index)

In [9]:
# Creating new column called "split" to decide which record goes into train or test
df_sort["split"] = df_sort.groupby("AnimalId", group_keys=False).apply(split_by_time)

/var/folders/5n/f_9m5grn7vq6mx7m6n8bwb6c0000gn/T/ipykernel_1806/2498999006.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sort["split"] = df_sort.groupby("AnimalId", group_keys=False).apply(split_by_time)


In [10]:
# Creating training and testing sets
df_train = df_sort[df_sort["split"] == "train"]
df_test = df_sort[df_sort["split"] == "test"]

In [11]:
# Checking
print(df_train.shape, df_test.shape)
print(df_train["AnimalId"].nunique(), df_test["AnimalId"].nunique())

(249625, 15) (62181, 15)
448 448


# Model building and hyperparameter tuning

In [12]:
# Creating features and labels
features = ["AvgMilkFlow", "Flow30_60Session", "YieldFirst2Min_Session",
            "YieldSession", "DurationSession_Sec", "Milking"]

X_train = df_train[features]
y_train = df_train["AnimalId"]

X_test = df_test[features]
y_test = df_test["AnimalId"]

In [17]:
# 1. Define the ranges to search over
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(5, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}

# 2. Base model — single-threaded, since the search itself will parallelize
rf = RandomForestClassifier(random_state=42, n_jobs=1)

# 3. Wrap it in RandomizedSearchCV
rand_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,         # how many random combinations to try
    cv=3,               # 3-fold cross-validation per combination
    scoring='accuracy',
    n_jobs=2,           # modest parallelism, not -1, to avoid memory spikes
    random_state=0,
    #verbose=2            # prints progress as it runs
)

# 4. Run the search on your training data
rand_search.fit(X_train, y_train)

# 5. See what it found
print("Best params:", rand_search.best_params_)
print("Best CV accuracy:", rand_search.best_score_)

# 6. Pull out the best model, already trained on the full training set
best_rf = rand_search.best_estimator_

/Users/muyuanli/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'max_depth': 14, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 9, 'n_estimators': 164}
Best CV accuracy: 0.04392189739726996


In [18]:
# Check feature importance
importances = pd.Series(best_rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

YieldFirst2Min_Session    0.234046
Flow30_60Session          0.197654
YieldSession              0.194613
DurationSession_Sec       0.193598
AvgMilkFlow               0.127227
Milking                   0.052862
dtype: float64


In [19]:
# Droping "Milking"
features_trimmed = ["AvgMilkFlow", "Flow30_60Session", "YieldFirst2Min_Session",
                     "YieldSession", "DurationSession_Sec"]
X_train_trimmed = df_train[features_trimmed]
X_test_trimmed = df_test[features_trimmed]

In [20]:
# Repeat randomized search process without milking 
# 1. Define the ranges to search over
param_dist_2 = {
    'n_estimators': randint(100, 200),
    'max_depth': randint(10, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2']
}

# 2. Base model — single-threaded, since the search itself will parallelize
rf_2 = RandomForestClassifier(random_state=42, n_jobs=1)

# 3. Wrap it in RandomizedSearchCV
rand_search_2 = RandomizedSearchCV(
    estimator=rf_2,
    param_distributions=param_dist_2,
    n_iter=10,         # how many random combinations to try
    cv=3,               # 3-fold cross-validation per combination
    scoring='accuracy',
    n_jobs=2,           # modest parallelism, not -1, to avoid memory spikes
    random_state=0,
    #verbose=2            # prints progress as it runs
)

# 4. Run the search on your training data
rand_search_2.fit(X_train_trimmed, y_train)

# 5. See what it found
print("Best params:", rand_search_2.best_params_)
print("Best CV accuracy:", rand_search_2.best_score_)

# 6. Pull out the best model, already trained on the full training set
best_rf_2 = rand_search_2.best_estimator_

/Users/muyuanli/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'max_depth': 13, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 199}
Best CV accuracy: 0.043917893249829414


In [ ]:
best_index = rand_search_2.best_index_
mean_score = rand_search_2.cv_results_['mean_test_score'][best_index]
std_score = rand_search_2.cv_results_['std_test_score'][best_index]

print(f"CV F1-score: {mean_score:.3f} ± {std_score:.3f}")

In [21]:
# Testing
y_pred_tuned = best_rf_2.predict(X_test_trimmed)
tuned_acc = accuracy_score(y_test, y_pred_tuned)
print("Tuned top-1 accuracy:", tuned_acc)

# Top-5 accuracy, same method as before
proba = best_rf_2.predict_proba(X_test_trimmed)
classes = best_rf_2.classes_
top5_idx = np.argsort(proba, axis=1)[:, -5:]
top5_preds = classes[top5_idx]
top5_acc = np.mean([y_test.values[i] in top5_preds[i] for i in range(len(y_test))])
print("Tuned top-5 accuracy:", top5_acc)

Tuned top-1 accuracy: 0.035911291230440165
Tuned top-5 accuracy: 0.13063475981409112


In [22]:
cm = confusion_matrix(y_test, y_pred_tuned)
ConfusionMatrixDisplay(confusion_matrix=cm).plot();

Error in callback <function flush_figures at 0x125bc1ee0> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [ ]:
precision_score
recall_score